# Evolution: Education & Outreach — Thematic Analysis

Clusters all 666 articles in the EEO journal into themes using **BERTopic**:
sentence-transformer embeddings + UMAP dimensionality reduction + HDBSCAN clustering.

Run **Cleaning.ipynb** first to generate `EEO_articles_clean.csv`.


In [ ]:
import ast
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from umap import UMAP

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


In [ ]:
df = pd.read_csv("EEO_articles_clean.csv")
df["Published date"] = pd.to_datetime(df["Published date"])
df["Year"] = df["Published date"].dt.year
df["Authors"] = df["Authors"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

print(f"{len(df)} articles, {df['Year'].min()}–{df['Year'].max()}")
df.head(3)

## Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df.groupby("Year").size().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Articles Published per Year", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["Content Type"].value_counts().plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Articles by Content Type", fontsize=13)
ax.set_xlabel("Count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
author_counts = Counter(a for authors in df["Authors"] for a in authors)
top_authors = pd.Series(author_counts).nlargest(15)

fig, ax = plt.subplots(figsize=(8, 5))
top_authors.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Top 15 Most Published Authors", fontsize=13)
ax.set_xlabel("Articles")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Topic Modelling with BERTopic

Article titles are used for embedding (clean signal, no boilerplate).
Full article text drives keyword extraction for each topic.


In [ ]:
titles = df["Title"].tolist()
docs = df["Article text"].fillna("").tolist()

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(titles, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

In [ ]:
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=8,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True,
)

# Fit on full article text, pass pre-computed title embeddings for clustering
topics, probs = topic_model.fit_transform(docs, embeddings)
df["Topic"] = topics

In [ ]:
topic_info = topic_model.get_topic_info()
print(f"{len(topic_info) - 1} topics found (plus outlier bucket -1)")
topic_info

## Visualizations

In [ ]:
# Top keywords per topic
topic_model.visualize_barchart(top_n_topics=15, n_words=8)

In [ ]:
# 2-D topic map
topic_model.visualize_topics()

## Topics Over Time

How each theme has evolved across the journal's history (2007–2023).


In [ ]:
timestamps = df["Published date"].tolist()
topics_over_time = topic_model.topics_over_time(docs, timestamps, nr_bins=10)
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)